# Chuẩn bị dữ liệu dự báo PM2.5 đa chân trời

Notebook này xây dựng đồng thời 24 biến mục tiêu, từ PM2.5 tại `t+1` đến `t+24`.
Mục tiêu được ghép bằng khóa `(city, timestamp)` thay vì dịch theo số dòng, do đó một
timestamp bị thiếu không làm target bị lệch giờ.

Đây là bài toán **hồi quy đa chân trời**. Tại thời điểm `t`, mô hình chỉ được sử dụng
thông tin đã quan sát đến `t`; các giá trị PM2.5 tương lai chỉ xuất hiện trong target.


In [1]:
from pathlib import Path
import json
import random
from typing import Any

import joblib
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False,
})
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Nhận diện project bằng cấu trúc, không phụ thuộc tên thư mục tạm của project.
CURRENT_DIR = Path.cwd().resolve()
SEARCH_DIRS = [CURRENT_DIR, CURRENT_DIR.parent]
SEARCH_DIRS.extend(path for path in CURRENT_DIR.iterdir() if path.is_dir())
PROJECT_CANDIDATES = []
for candidate in SEARCH_DIRS:
    data_file = candidate / "data" / "processed" / "pm25_training_data_enriched.csv"
    notebook_marker = candidate / "model" / "0_multihorizon_data_preparation.ipynb"
    if notebook_marker.exists() and data_file.exists():
        resolved = candidate.resolve()
        if resolved not in PROJECT_CANDIDATES:
            PROJECT_CANDIDATES.append(resolved)

if len(PROJECT_CANDIDATES) == 1:
    PROJECT_ROOT = PROJECT_CANDIDATES[0]
elif not PROJECT_CANDIDATES:
    raise FileNotFoundError(
        "Không tìm thấy project chứa đồng thời model và "
        "data/processed/pm25_training_data_enriched.csv."
    )
else:
    raise RuntimeError(
        "Có nhiều project phù hợp; hãy mở Jupyter tại đúng thư mục gốc cần chạy: "
        + ", ".join(str(path) for path in PROJECT_CANDIDATES)
    )

MODEL_DIR = PROJECT_ROOT / "model"
RESULTS_DIR = MODEL_DIR / "results"
CANDIDATES_DIR = MODEL_DIR / "candidates"
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "pm25_training_data_enriched.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATES_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS = np.arange(1, 25, dtype=int)
TARGET_COLUMNS = [f"target_pm25_t_plus_{h}" for h in HORIZONS]
MAX_HORIZON = int(HORIZONS.max())

POLLUTANT_FEATURES = ["pm25", "pm10", "o3", "no2", "so2", "co"]
WEATHER_FEATURES = [
    "temp", "humidity", "wind_speed", "wind_dir", "precip", "pressure", "cloud_cover"
]
TEMPORAL_FEATURES = ["hour", "day_of_week", "month", "is_weekend", "day_of_year"]
HISTORY_FEATURES = [
    "pm25_lag_1h", "pm25_lag_3h", "pm25_lag_6h", "pm25_lag_12h",
    "pm25_lag_24h", "pm25_lag_48h", "pm25_lag_72h", "pm25_lag_96h",
    "pm25_lag_120h", "pm25_lag_144h", "pm25_lag_168h",
    "pm25_roll_6h", "pm25_roll_12h", "pm25_roll_24h", "pm25_roll_72h",
    "pm25_roll_168h", "pm25_std_6h", "pm25_std_12h", "pm25_std_24h",
    "pm25_std_72h", "pm25_std_168h", "pm25_min_24h", "pm25_max_24h",
    "pm25_delta_1h", "pm25_delta_3h", "pm25_delta_24h",
    "pm25_roll_ratio_6h_24h", "pm25_roll_ratio_24h_72h",
    "pm25_same_hour_mean_7d", "pm25_same_hour_median_7d",
    "pm25_same_hour_std_7d", "pm25_same_hour_min_7d",
    "pm25_same_hour_max_7d", "pm25_same_hour_ratio_7d", "pm25_weekly_delta",
]
ENGINEERED_FEATURES = [
    "ventilation_index", "humid_stagnation", "rain_flag", "calm_wind",
    "high_humidity", "wind_x", "wind_y", "hour_sin", "hour_cos",
    "day_sin", "day_cos", "month_sin", "month_cos", "pm25_pm10_ratio",
    "no2_co_ratio",
]
NUMERIC_FEATURES = (
    POLLUTANT_FEATURES + WEATHER_FEATURES + TEMPORAL_FEATURES
    + HISTORY_FEATURES + ENGINEERED_FEATURES
)
CATEGORICAL_FEATURES = ["city", "season"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data: {DATA_PATH}")


Project root: D:\Project123456\aqi-vietnam\aqi-vietnam4
Data: D:\Project123456\aqi-vietnam\aqi-vietnam4\data\processed\pm25_training_data_enriched.csv


## Quy tắc tạo target và chia dữ liệu

Mỗi origin có 24 target. Chỉ giữ origin có đủ cả 24 timestamp tương lai và đủ feature.
Dữ liệu được chia tuần tự 70/15/15 theo thời gian kết thúc cửa sổ dự báo, kèm khoảng
purge 24 giờ. Không dùng random split vì sẽ làm rò rỉ cấu trúc thời gian.


In [2]:
def make_multihorizon_frame(data: pd.DataFrame) -> pd.DataFrame:
    """Ghép chính xác PM2.5 tại t+1,...,t+24 theo city và timestamp."""
    frame = data.copy()
    frame["datetime"] = pd.to_datetime(frame["datetime"])
    frame = frame.sort_values(["city", "datetime"]).reset_index(drop=True)
    if frame.duplicated(["city", "datetime"]).any():
        raise ValueError("Dữ liệu có city/datetime trùng, không thể ghép target chính xác.")

    lookup = frame.set_index(["city", "datetime"])["pm25"]
    for horizon, column in zip(HORIZONS, TARGET_COLUMNS):
        keys = pd.MultiIndex.from_arrays(
            [frame["city"], frame["datetime"] + pd.to_timedelta(horizon, unit="h")],
            names=["city", "datetime"],
        )
        frame[column] = lookup.reindex(keys).to_numpy(dtype=float)
    frame["forecast_end"] = frame["datetime"] + pd.Timedelta(hours=MAX_HORIZON)
    return frame


def make_temporal_split(
    frame: pd.DataFrame,
    train_fraction: float = 0.70,
    validation_fraction: float = 0.15,
    purge_hours: int = MAX_HORIZON,
) -> dict[str, Any]:
    """Chia theo forecast_end để không có cửa sổ target giao nhau giữa các tập."""
    unique_ends = pd.Series(frame["forecast_end"].dropna().sort_values().unique())
    train_cut = pd.Timestamp(unique_ends.iloc[int(len(unique_ends) * train_fraction)])
    val_cut = pd.Timestamp(
        unique_ends.iloc[int(len(unique_ends) * (train_fraction + validation_fraction))]
    )
    purge = pd.Timedelta(hours=purge_hours)
    forecast_end = pd.to_datetime(frame["forecast_end"])
    masks = {
        "train": forecast_end < train_cut,
        "validation": (forecast_end >= train_cut + purge) & (forecast_end < val_cut),
        "test": forecast_end >= val_cut + purge,
    }
    if any(not mask.any() for mask in masks.values()):
        raise ValueError("Temporal split tạo ra ít nhất một tập rỗng.")
    return {
        "masks": masks,
        "train_cut": train_cut,
        "val_cut": val_cut,
        "purge_hours": purge_hours,
    }


def split_summary(frame: pd.DataFrame, split: dict[str, Any]) -> dict[str, Any]:
    summary: dict[str, Any] = {
        "strategy": "global chronological 70/15/15 by forecast_end with 24-hour purge gaps",
        "train_cut": split["train_cut"].isoformat(),
        "validation_cut": split["val_cut"].isoformat(),
        "purge_hours": int(split["purge_hours"]),
        "horizons": HORIZONS.tolist(),
    }
    for name, mask in split["masks"].items():
        part = frame.loc[mask]
        summary[name] = {
            "rows": int(len(part)),
            "source_start": part["datetime"].min().isoformat(),
            "source_end": part["datetime"].max().isoformat(),
            "forecast_end_start": part["forecast_end"].min().isoformat(),
            "forecast_end_end": part["forecast_end"].max().isoformat(),
        }
    return summary


def load_model_frame() -> tuple[pd.DataFrame, dict[str, Any], pd.DataFrame]:
    raw = pd.read_csv(DATA_PATH, low_memory=False)
    raw["datetime"] = pd.to_datetime(raw["datetime"])
    supervised = make_multihorizon_frame(raw)
    required = NUMERIC_FEATURES + CATEGORICAL_FEATURES + TARGET_COLUMNS
    missing_columns = sorted(set(required) - set(supervised.columns))
    if missing_columns:
        raise KeyError(f"Thiếu cột cần thiết: {missing_columns}")
    frame = supervised.dropna(subset=required).copy().reset_index(drop=True)
    split = make_temporal_split(frame)
    manifest = split_summary(frame, split)
    (RESULTS_DIR / "multihorizon_temporal_split.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return frame, split, raw


def build_feature_matrix(
    frame: pd.DataFrame,
    feature_columns: list[str] | None = None,
) -> pd.DataFrame:
    matrix = pd.get_dummies(
        frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES],
        columns=CATEGORICAL_FEATURES,
        drop_first=False,
        dtype=float,
    )
    if feature_columns is None:
        return matrix.astype(np.float32)
    for column in feature_columns:
        if column not in matrix:
            matrix[column] = 0.0
    return matrix.reindex(columns=feature_columns, fill_value=0.0).astype(np.float32)


def regression_metrics(actual: Any, predicted: Any) -> dict[str, float]:
    y_true = np.asarray(actual, dtype=float)
    y_pred = np.clip(np.asarray(predicted, dtype=float), 0.0, None)
    return {
        "rmse_ug_m3": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae_ug_m3": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "bias_ug_m3": float(np.mean(y_pred - y_true)),
    }


def prediction_frame(
    frame: pd.DataFrame,
    mask: pd.Series,
    predictions: np.ndarray,
    model_name: str,
    split_name: str,
) -> pd.DataFrame:
    part = frame.loc[mask].reset_index(drop=True)
    actual = part[TARGET_COLUMNS].to_numpy(dtype=float)
    predicted = np.clip(np.asarray(predictions, dtype=float), 0.0, None)
    if predicted.shape != actual.shape:
        raise ValueError(f"Prediction shape {predicted.shape} khác target shape {actual.shape}.")
    rows = len(part)
    output = pd.DataFrame({
        "model": model_name,
        "split": split_name,
        "city": np.repeat(part["city"].to_numpy(), len(HORIZONS)),
        "source_time": np.repeat(part["datetime"].to_numpy(), len(HORIZONS)),
        "horizon": np.tile(HORIZONS, rows),
        "actual_pm25": actual.reshape(-1),
        "predicted_pm25": predicted.reshape(-1),
    })
    output["target_time"] = pd.to_datetime(output["source_time"]) + pd.to_timedelta(
        output["horizon"], unit="h"
    )
    output["abs_error_ug_m3"] = np.abs(output["actual_pm25"] - output["predicted_pm25"])
    return output


def metric_tables(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    horizon_rows = []
    for (model, split_name, horizon), part in predictions.groupby(
        ["model", "split", "horizon"], sort=True
    ):
        horizon_rows.append({
            "model": model,
            "split": split_name,
            "horizon": int(horizon),
            "rows": int(len(part)),
            **regression_metrics(part["actual_pm25"], part["predicted_pm25"]),
        })
    by_horizon = pd.DataFrame(horizon_rows)

    city_rows = []
    for (model, split_name, city), part in predictions.groupby(
        ["model", "split", "city"], sort=True
    ):
        city_rows.append({
            "model": model,
            "split": split_name,
            "city": city,
            "rows": int(len(part)),
            **regression_metrics(part["actual_pm25"], part["predicted_pm25"]),
        })
    by_city = pd.DataFrame(city_rows)

    summary_rows = []
    for (model, split_name), part in predictions.groupby(["model", "split"], sort=True):
        horizon_part = by_horizon.loc[
            by_horizon["model"].eq(model) & by_horizon["split"].eq(split_name)
        ]
        summary_rows.append({
            "model": model,
            "split": split_name,
            "rows": int(len(part)),
            "mean_horizon_rmse_ug_m3": float(horizon_part["rmse_ug_m3"].mean()),
            "mean_horizon_mae_ug_m3": float(horizon_part["mae_ug_m3"].mean()),
            **{f"global_{key}": value for key, value in regression_metrics(
                part["actual_pm25"], part["predicted_pm25"]
            ).items()},
        })
    return by_horizon, by_city, pd.DataFrame(summary_rows)


def save_evaluation(model_slug: str, predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    by_horizon, by_city, summary = metric_tables(predictions)
    predictions.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )
    by_horizon.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_by_horizon.csv",
        index=False,
        encoding="utf-8-sig",
    )
    by_city.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_by_city.csv",
        index=False,
        encoding="utf-8-sig",
    )
    summary.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )
    return by_horizon, by_city, summary


In [3]:
model_frame, split, raw_data = load_model_frame()
manifest = split_summary(model_frame, split)
display(pd.DataFrame({
    name: {
        "Số origin": values["rows"],
        "Bắt đầu origin": values["source_start"],
        "Kết thúc origin": values["source_end"],
    }
    for name, values in manifest.items()
    if isinstance(values, dict) and "rows" in values
}).T)
print(f"Tổng số origin hợp lệ: {len(model_frame):,}")
print(f"Số target trên mỗi origin: {len(TARGET_COLUMNS)}")


,Số origin,Bắt đầu origin,Kết thúc origin
train,68829,2022-08-12T07:00:00,2025-03-25T05:00:00
validation,14679,2025-03-26T06:00:00,2025-10-16T02:00:00
test,14679,2025-10-17T03:00:00,2026-05-08T23:00:00


Tổng số origin hợp lệ: 98,331
Số target trên mỗi origin: 24


In [4]:
coverage = []
supervised = make_multihorizon_frame(raw_data)
for horizon, column in zip(HORIZONS, TARGET_COLUMNS):
    coverage.append({
        "horizon": int(horizon),
        "target_coverage": float(supervised[column].notna().mean()),
        "missing_targets": int(supervised[column].isna().sum()),
    })
coverage = pd.DataFrame(coverage)
display(coverage)

axis = coverage.plot(
    x="horizon", y="target_coverage", marker="o", figsize=(9, 4), legend=False
)
axis.set_xlabel("Chân trời dự báo (giờ)")
axis.set_ylabel("Tỷ lệ target hợp lệ")
axis.set_ylim(0.95, 1.001)
axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()


,horizon,target_coverage,missing_targets
0,1,0.999970,3
1,2,0.999939,6
2,3,0.999909,9
3,4,0.999879,12
4,5,0.999848,15
5,6,0.999818,18
6,7,0.999788,21
7,8,0.999757,24
8,9,0.999727,27
9,10,0.999697,30


C:\Users\nguyen\AppData\Local\Temp\ipykernel_7952\1592480189.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Kiểm tra chống rò rỉ

Các assertion dưới đây xác nhận target cuối cùng luôn đúng `t+24`, mọi target thuộc
Train kết thúc trước Validation, và các tập được ngăn bởi purge gap đã khai báo.


In [5]:
sample = model_frame.sample(min(500, len(model_frame)), random_state=SEED)
lookup = raw_data.set_index(["city", "datetime"])["pm25"]
for horizon, column in zip(HORIZONS, TARGET_COLUMNS):
    keys = pd.MultiIndex.from_arrays([
        sample["city"], sample["datetime"] + pd.to_timedelta(horizon, unit="h")
    ])
    expected = lookup.reindex(keys).to_numpy(dtype=float)
    np.testing.assert_allclose(sample[column].to_numpy(dtype=float), expected)

train_end = model_frame.loc[split["masks"]["train"], "forecast_end"].max()
val_start = model_frame.loc[split["masks"]["validation"], "forecast_end"].min()
val_end = model_frame.loc[split["masks"]["validation"], "forecast_end"].max()
test_start = model_frame.loc[split["masks"]["test"], "forecast_end"].min()
assert train_end < val_start < val_end < test_start
assert val_start - train_end >= pd.Timedelta(hours=MAX_HORIZON)
assert test_start - val_end >= pd.Timedelta(hours=MAX_HORIZON)
print("Đã xác nhận target t+1...t+24 và temporal split không giao nhau.")


Đã xác nhận target t+1...t+24 và temporal split không giao nhau.
